In [1]:
from google.colab import drive
import os
import zipfile
import sys
import json
import pandas as pd
import tqdm

In [7]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set path to Google Drive project directory
drive_project_dir = '/content/drive/MyDrive/twitter-multimodal-hate-speech-classifier'


# 3. Check if the project is already cloned; if not, clone it automatically
if not drive_project_dir.exists():
    print("Cloning project into Google Drive...")
    repo_url = "https://github.com/mega_herz/twitter-multimodal-hate-speech-classifier.git"
    !git clone {repo_url} {project_dir}
else:
    print("Project already exists in Google Drive. Pulling latest updates...")
    os.chdir(drive_project_dir)
    !git pull


# Set project directory as current working directory
os.chdir(drive_project_dir)

# Set path to and, if needed, create target directory for data files in Google Drive
drive_data_dir = 'data' # relative path
os.makedirs(drive_data_dir, exist_ok=True)

# Path to local fast storage where raw data is stored
raw_data_dir = '/content/local_dataset'
os.makedirs(raw_data_dir, exist_ok=True)

In [6]:
proba_dir = 'proba'
os.makedirs(proba_dir, exist_ok=True)

In [4]:
# Import functions from .py file
from src.load_utils import load_data_from_json, load_image_texts
from src.data_utils import extract_timestamp

## 1.Download dataset (.zip) from Kaggle into Google Drive (requires Kaggle API token)

In [5]:
# Download dataset .zip from Kaggle to current working directory's subfolder 'data' (Google Drive)
!kaggle datasets download -d victorcallejasf/multimodal-hate-speech -p data

Dataset URL: https://www.kaggle.com/datasets/victorcallejasf/multimodal-hate-speech
License(s): copyright-authors
100% 5.97G/5.97G [01:03<00:00, 102MB/s]



## 2.Unzip data  from Google Drive into Colab runtime

In [5]:
# Path to raw data .zip file
zip_path = os.path.join(drive_data_dir, 'multimodal-hate-speech.zip')

# Unzip raw data from Google Drive to temporary local Colab fast storage
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(raw_data_dir)

In [14]:
# Path to JSON file with data
json_file_path = os.path.join(raw_data_dir, 'MMHS150K_GT.json')

In [15]:
# Read JSON and load data as dataframe
raw_df = load_data_from_json(json_file_path)

raw_df

,file_id,img_url,labels,tweet_url,tweet_text,labels_str
0,1114679353714016256,http://pbs.twimg.com/tweet_video_thumb/D3gi9MH...,"[4, 1, 3]",https://twitter.com/user/status/11146793537140...,@FriskDontMiss Nigga https://t.co/cAsaLWEpue,"[Religion, Racist, Homophobe]"
1,1063020048816660480,http://pbs.twimg.com/ext_tw_video_thumb/106301...,"[5, 5, 5]",https://twitter.com/user/status/10630200488166...,My horses are retarded https://t.co/HYhqc6d5WN,"[OtherHate, OtherHate, OtherHate]"
2,1108927368075374593,http://pbs.twimg.com/media/D2OzhzHUwAADQjd.jpg,"[0, 0, 0]",https://twitter.com/user/status/11089273680753...,“NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL S...,"[NotHate, NotHate, NotHate]"
3,1114558534635618305,http://pbs.twimg.com/ext_tw_video_thumb/111401...,"[1, 0, 0]",https://twitter.com/user/status/11145585346356...,RT xxSuGVNGxx: I ran into this HOLY NIGGA TODA...,"[Racist, NotHate, NotHate]"
4,1035252480215592966,http://pbs.twimg.com/media/Dl30pGIU8AAVGxO.jpg,"[1, 0, 1]",https://twitter.com/user/status/10352524802155...,“EVERYbody calling you Nigger now!” https://t....,"[Racist, NotHate, Racist]"
...,...,...,...,...,...,...
149818,1114170734472048640,http://pbs.twimg.com/tweet_video_thumb/D3ZUXNw...,"[2, 5, 0]",https://twitter.com/user/status/11141707344720...,@svdate @gtconway3d I would just say hes Donny...,"[Sexist, OtherHate, NotHate]"
149819,1110368198786846720,http://pbs.twimg.com/ext_tw_video_thumb/111036...,"[0, 0, 0]",https://twitter.com/user/status/11103681987868...,@Cheftime_Dev congrats my nigga keep on grindi...,"[NotHate, NotHate, NotHate]"
149820,1106941858540851200,http://pbs.twimg.com/media/D1yluGmXgAEKNG5.jpg,"[0, 1, 0]",https://twitter.com/user/status/11069418585408...,My nigga big shitty https://t.co/e0snJGBgH9,"[NotHate, Racist, NotHate]"
149821,1105268309233188865,http://pbs.twimg.com/tweet_video_thumb/D1azqiz...,"[1, 0, 0]",https://twitter.com/user/status/11052683092331...,did she just say “my nigga” to Rich? &amp; she...,"[Racist, NotHate, NotHate]"


## 3.Derive Timestamp of messages from their IDs and add it as a column to the dataset

In [16]:
raw_df['created_at'] = extract_timestamp(raw_df['file_id'])

In [17]:
raw_df.head(3)

,file_id,img_url,labels,tweet_url,tweet_text,labels_str,created_at
0,1114679353714016256,http://pbs.twimg.com/tweet_video_thumb/D3gi9MH...,"[4, 1, 3]",https://twitter.com/user/status/11146793537140...,@FriskDontMiss Nigga https://t.co/cAsaLWEpue,"[Religion, Racist, Homophobe]",2019-04-07 00:00:42.323
1,1063020048816660480,http://pbs.twimg.com/ext_tw_video_thumb/106301...,"[5, 5, 5]",https://twitter.com/user/status/10630200488166...,My horses are retarded https://t.co/HYhqc6d5WN,"[OtherHate, OtherHate, OtherHate]",2018-11-15 10:45:04.252
2,1108927368075374593,http://pbs.twimg.com/media/D2OzhzHUwAADQjd.jpg,"[0, 0, 0]",https://twitter.com/user/status/11089273680753...,“NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL S...,"[NotHate, NotHate, NotHate]",2019-03-22 03:04:22.080


## 4.Add text from image (posted in tweet) which was previously extracted using OCR

In [32]:
# Load text data into a dictionary
text_dir = os.path.join(raw_data_dir, 'img_txt')
text_data = load_image_texts(text_dir)

In [35]:
# Map the extracted text to DataFrame using file_id
raw_df['img_text'] = raw_df['file_id'].astype(str).map(text_data).fillna("")
raw_df

,file_id,img_url,labels,tweet_url,tweet_text,labels_str,created_at,img_text
0,1114679353714016256,http://pbs.twimg.com/tweet_video_thumb/D3gi9MH...,"[4, 1, 3]",https://twitter.com/user/status/11146793537140...,@FriskDontMiss Nigga https://t.co/cAsaLWEpue,"[Religion, Racist, Homophobe]",2019-04-07 00:00:42.323,#YOUNGERU SAVE IT
1,1063020048816660480,http://pbs.twimg.com/ext_tw_video_thumb/106301...,"[5, 5, 5]",https://twitter.com/user/status/10630200488166...,My horses are retarded https://t.co/HYhqc6d5WN,"[OtherHate, OtherHate, OtherHate]",2018-11-15 10:45:04.252,
2,1108927368075374593,http://pbs.twimg.com/media/D2OzhzHUwAADQjd.jpg,"[0, 0, 0]",https://twitter.com/user/status/11089273680753...,“NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL S...,"[NotHate, NotHate, NotHate]",2019-03-22 03:04:22.080,
3,1114558534635618305,http://pbs.twimg.com/ext_tw_video_thumb/111401...,"[1, 0, 0]",https://twitter.com/user/status/11145585346356...,RT xxSuGVNGxx: I ran into this HOLY NIGGA TODA...,"[Racist, NotHate, NotHate]",2019-04-06 16:00:36.810,
4,1035252480215592966,http://pbs.twimg.com/media/Dl30pGIU8AAVGxO.jpg,"[1, 0, 1]",https://twitter.com/user/status/10352524802155...,“EVERYbody calling you Nigger now!” https://t....,"[Racist, NotHate, Racist]",2018-08-30 19:46:40.001,
...,...,...,...,...,...,...,...,...
149818,1114170734472048640,http://pbs.twimg.com/tweet_video_thumb/D3ZUXNw...,"[2, 5, 0]",https://twitter.com/user/status/11141707344720...,@svdate @gtconway3d I would just say hes Donny...,"[Sexist, OtherHate, NotHate]",2019-04-05 14:19:38.046,LATE MOGIF LATE MOTIV
149819,1110368198786846720,http://pbs.twimg.com/ext_tw_video_thumb/111036...,"[0, 0, 0]",https://twitter.com/user/status/11103681987868...,@Cheftime_Dev congrats my nigga keep on grindi...,"[NotHate, NotHate, NotHate]",2019-03-26 02:29:42.891,ON AIR Elapsed Time: 05.47:18 Select Your Leve...
149820,1106941858540851200,http://pbs.twimg.com/media/D1yluGmXgAEKNG5.jpg,"[0, 1, 0]",https://twitter.com/user/status/11069418585408...,My nigga big shitty https://t.co/e0snJGBgH9,"[NotHate, Racist, NotHate]",2019-03-16 15:34:39.718,
149821,1105268309233188865,http://pbs.twimg.com/tweet_video_thumb/D1azqiz...,"[1, 0, 0]",https://twitter.com/user/status/11052683092331...,did she just say “my nigga” to Rich? &amp; she...,"[Racist, NotHate, NotHate]",2019-03-12 00:44:34.470,


## 5.Save data as .parquet on Google Drive for further work

In [19]:
# Save to data Google Drive for later use
raw_df.to_parquet(os.path.join(drive_data_dir, 'twitter_data_with_text_from_pics.parquet'))